In [1]:
import os

os.makedirs("data/raw/location", exist_ok=True)

In [1]:
import requests

API_KEY = "38ca35795bee28d618f1d07252a3eb39a912f8ea"  # confirm: is this literally still the placeholder in your real script?

feed_id = "20422"
url = f"https://data.bus-data.dft.gov.uk/api/v1/datafeed/{feed_id}/?api_key={API_KEY}"

response = requests.get(url)

print("Status code:", response.status_code)
print("Headers:", dict(response.headers))
print("Content length:", len(response.content))
print("Raw content (first 500 bytes):", response.content[:500])

Status code: 200
Headers: {'Content-Type': 'text/xml', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Date': 'Wed, 22 Jul 2026 14:42:57 GMT', 'content-encoding': 'gzip', 'Content-Security-Policy': "default-src 'self'; frame-ancestors 'self'; connect-src 'self' *.mapbox.com *.localhost:8000 *.bus-data.dft.gov.uk google-analytics.com; style-src 'self' 'unsafe-inline'; img-src 'self' data: blob:; worker-src 'self' blob:; font-src 'self'; script-src 'self' 'unsafe-inline' www.googletagmanager.com ajax.googleapis.com/ajax/libs/jquery google-analytics.com; object-src 'none'", 'Set-Cookie': 'AWSALB=WidTfa0+oHx0IE/hroRAHtjNhCF9QqSO3geEGlJG2qRim2KKWmPvcyRVI7peVC5b1g7sDCtPJqWJ3QWM61cwmko/EI0mUckRlC6yiC5Hap15q8P2aOs3VZ//KUuN; Expires=Wed, 29 Jul 2026 14:42:57 GMT; Path=/, AWSALBCORS=WidTfa0+oHx0IE/hroRAHtjNhCF9QqSO3geEGlJG2qRim2KKWmPvcyRVI7peVC5b1g7sDCtPJqWJ3QWM61cwmko/EI0mUckRlC6yiC5Hap15q8P2aOs3VZ//KUuN; Expires=Wed, 29 Jul 2026 14:42:57 GMT; Path=/; SameSite=None; Secure', 'Serve

In [2]:
import requests

API_KEY = "38ca35795bee28d618f1d07252a3eb39a912f8ea"

FEED_IDS = ["20422", "18880", "16387", "14336", "14327", "12880"]
BASE_URL = "https://data.bus-data.dft.gov.uk/api/v1/datafeed/{}/?api_key={}"

for feed_id in FEED_IDS:
    url = BASE_URL.format(feed_id, API_KEY)
    r = requests.get(url, timeout=30)
    ok = "VehicleActivity" in r.text
    print(f"Feed {feed_id}: status={r.status_code}, size={len(r.content)} bytes, has_VehicleActivity={ok}")

Feed 20422: status=200, size=356561 bytes, has_VehicleActivity=True
Feed 18880: status=200, size=441262 bytes, has_VehicleActivity=True
Feed 16387: status=200, size=171364 bytes, has_VehicleActivity=True
Feed 14336: status=200, size=637648 bytes, has_VehicleActivity=True
Feed 14327: status=200, size=39714 bytes, has_VehicleActivity=True
Feed 12880: status=200, size=125919 bytes, has_VehicleActivity=True


In [ ]:
import requests
import time
import os
from datetime import datetime
import xml.etree.ElementTree as ET

API_KEY = "38ca35795bee28d618f1d07252a3eb39a912f8ea"

FEED_IDS = {
    "20422": "BNVB",
    "18880": "BNGN",
    "16387": "BNML",
    "14336": "BNSM",
    "14327": "BNFM",
    "12880": "BNDB",
}

BASE_URL = "https://data.bus-data.dft.gov.uk/api/v1/datafeed/{}/?api_key={}"
SAVE_PATH = "data/raw/location"
os.makedirs(SAVE_PATH, exist_ok=True)

NUM_SNAPSHOTS = 150          # ~5 hours if interval=120s -- adjust based on your time budget
INTERVAL_SECONDS = 120       # 2 minutes between rounds; change to 300 for 5-min spacing over a longer window

total_vehicle_count = 0

for i in range(NUM_SNAPSHOTS):
    round_count = 0
    print(f"\n--- Snapshot round {i+1}/{NUM_SNAPSHOTS} at {datetime.now().strftime('%H:%M:%S')} ---")

    for feed_id, feed_name in FEED_IDS.items():
        url = BASE_URL.format(feed_id, API_KEY)
        try:
            response = requests.get(url, timeout=30)
        except requests.RequestException as e:
            print(f"  {feed_name} ({feed_id}): REQUEST FAILED - {e}")
            continue

        if response.status_code != 200:
            print(f"  {feed_name} ({feed_id}): BAD STATUS {response.status_code} - {response.text[:150]}")
            continue

        if b"VehicleActivity" not in response.content:
            print(f"  {feed_name} ({feed_id}): NO VehicleActivity in response - skipping save")
            continue

        # count vehicles in this response for a live row estimate
        try:
            root = ET.fromstring(response.content)
            n_vehicles = len([e for e in root.iter() if e.tag.endswith("VehicleActivity")])
        except ET.ParseError:
            n_vehicles = 0

        round_count += n_vehicles

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{SAVE_PATH}/feed_{feed_id}_{timestamp}.xml"
        with open(filename, "wb") as f:
            f.write(response.content)

        print(f"  {feed_name} ({feed_id}): saved, {n_vehicles} vehicles -> {filename}")

    total_vehicle_count += round_count
    print(f"  Round total: {round_count} vehicles | Running total: {total_vehicle_count}")

    if i < NUM_SNAPSHOTS - 1:
        time.sleep(INTERVAL_SECONDS)

print(f"\n✅ Collection complete! Total vehicle records collected: {total_vehicle_count}")


--- Snapshot round 1/150 at 20:30:50 ---
  BNVB (20422): saved, 283 vehicles -> data/raw/location/feed_20422_20260722_203051.xml
  BNGN (18880): saved, 342 vehicles -> data/raw/location/feed_18880_20260722_203052.xml
  BNML (16387): saved, 130 vehicles -> data/raw/location/feed_16387_20260722_203053.xml
  BNSM (14336): saved, 492 vehicles -> data/raw/location/feed_14336_20260722_203053.xml
  BNFM (14327): saved, 30 vehicles -> data/raw/location/feed_14327_20260722_203053.xml
  BNDB (12880): saved, 98 vehicles -> data/raw/location/feed_12880_20260722_203054.xml
  Round total: 1375 vehicles | Running total: 1375
